# CiviLanka.Agent — Interactive Agentic RAG Testing Notebook

**Member 2 — Infrastructure & Asset Registry**  
**AI Agent:** Municipal Cost & Material Estimator Agent  
**Architecture:** Hybrid Search (Chroma Vector + BM25 with Reciprocal Rank Fusion) + Stateful LangGraph  
**Domain:** Sri Lanka Municipal Infrastructure & CIDA/BSR Construction Rates in LKR  

---

### What this notebook tests:
1. **Part 0**: Environment & dependency check
2. **Part 1**: Sri Lanka BSR Knowledge Base inspection
3. **Part 2**: Hybrid Search (BM25 vs Vector vs Reciprocal Rank Fusion)
4. **Part 3**: LangGraph Agentic RAG execution (Router → Retrieve → Grade → Rewrite → Estimate)
5. **Part 4**: Live FastAPI HTTP Endpoint Testing (`http://127.0.0.1:8001`)

## Part 0 — Environment & Import Check

In [1]:
import os
import sys
from pathlib import Path
import json
import requests

# Ensure CiviLanka.Agent directory is in python path
AGENT_DIR = Path.cwd().resolve()
if str(AGENT_DIR) not in sys.path:
    sys.path.insert(0, str(AGENT_DIR))

from agent.retriever import hybrid_search, search_formatted, get_vector_store, get_bm25_retriever
from agent.graph import create_cost_estimator_graph

print("Python Interpreter :", sys.executable)
print("Working Directory  :", AGENT_DIR)
print("Environment Check  : All agent modules imported successfully!")

Python Interpreter : D:\IT24103847_Infrastructure & Asset Registry\CiviLanka-AI\CiviLanka-AI\CiviLanka.Agent\.venv\Scripts\python.exe
Working Directory  : D:\IT24103847_Infrastructure & Asset Registry\CiviLanka-AI\CiviLanka-AI\CiviLanka.Agent
Environment Check  : All agent modules imported successfully!


## Part 1 — Inspect Sri Lanka BSR Knowledge Base
Check the loaded Sri Lankan municipal rate schedules and technical specifications in `data/`.

In [2]:
data_dir = AGENT_DIR / "data"
markdown_files = sorted(data_dir.glob("*.md"))

print(f"Found {len(markdown_files)} knowledge base files in {data_dir}:\n")
for mf in markdown_files:
    size_kb = mf.stat().st_size / 1024
    lines = len(mf.read_text(encoding='utf-8').splitlines())
    print(f"  - {mf.name:<30} ({size_kb:.1f} KB, {lines} lines)")

# Preview first 20 lines of Sri Lanka BSR rates
print("\n--- Sample Preview: sri_lanka_bsr_rates.md ---")
preview = (data_dir / "sri_lanka_bsr_rates.md").read_text(encoding='utf-8').splitlines()[:22]
print("\n".join(preview))

Found 3 knowledge base files in D:\IT24103847_Infrastructure & Asset Registry\CiviLanka-AI\CiviLanka-AI\CiviLanka.Agent\data:

  - emergency_response_guide.md    (2.9 KB, 55 lines)
  - municipal_repair_specs.md      (4.4 KB, 77 lines)
  - sri_lanka_bsr_rates.md         (6.2 KB, 117 lines)

--- Sample Preview: sri_lanka_bsr_rates.md ---
# Sri Lanka Municipal Infrastructure & Construction — Building Schedule of Rates (BSR 2024/2026)

Source: CIDA (Construction Industry Development Authority) & National Water Supply and Drainage Board (NWSDB) & Road Development Authority (RDA) Standards
All unit rates and costs are expressed in Sri Lankan Rupees (LKR).

---

## 1. Potable Water Supply & Pipe Repair (NWSDB Standards)

### Pipe Supply & Laying (including jointing and pressure testing)

- **110mm uPVC Pipe (Class 1000, 10 Bar)**: LKR 4,850 per linear meter.
- **160mm uPVC Pipe (Class 1000, 10 Bar)**: LKR 7,600 per linear meter.
- **225mm uPVC Pipe (Class 1000, 10 Bar)**: LKR 12,200 per linea

## Part 2 — Hybrid Search (BM25 + Chroma Vector with Reciprocal Rank Fusion)

Hybrid search fuses two distinct retrieval mechanisms:
- **BM25 (Sparse)**: Matches exact engineering terms (e.g., `"110mm uPVC"`, `"CSS-1h tack coat"`, `"JCB Backhoe"`).
- **Vector Search (Dense)**: Matches conceptual meaning and defect descriptions.
- **Reciprocal Rank Fusion (RRF)**: Fuses both rankings using score $= \sum \frac{1}{60 + \text{rank}}$.

In [3]:
test_queries = [
    "110mm uPVC water pipe burst repair clamp cost",
    "asphalt pothole cold mix patching rate per m2",
    "drainage silt dredging and hume pipe replacement",
    "skilled pipe fitter and mason daily wage rate"
]

for query in test_queries:
    print("=" * 75)
    print(f"QUERY: {query}")
    print("=" * 75)
    hits = hybrid_search(query, k=2)
    for i, doc in enumerate(hits, 1):
        src = doc.metadata.get("source", "sri-lanka-bsr")
        clean_text = doc.page_content.replace('\n', ' ')[:130]
        print(f"  [Hit {i}] Source: {src}")
        print(f"         {clean_text}...")
    print()

QUERY: 110mm uPVC water pipe burst repair clamp cost
  [Hit 1] Source: municipal_repair_specs
         - **uPVC Pipe Circumferential Crack / Small Pinhole (< 50mm)**:   - Clean pipe exterior down to bare plastic.   - Install a stainl...
  [Hit 2] Source: emergency_response_guide
         ## 2. Standard Cost Estimation Formula for Municipal Repairs  The total estimated repair cost for any municipal infrastructure wor...

QUERY: asphalt pothole cold mix patching rate per m2
  [Hit 1] Source: sri_lanka_bsr_rates
         ## 2. Roadways, Pavements & Pothole Patching (RDA & CMC Rates)  ### Asphalt Concrete Paving & Pothole Repairs  - **Cold-mix asphal...
  [Hit 2] Source: municipal_repair_specs
         ## 1. Pothole & Road Surface Repair Standard Procedure  ### Step 1: Defect Marking & Perimeter Cutting  - Mark a rectangular bound...

QUERY: drainage silt dredging and hume pipe replacement
  [Hit 1] Source: municipal_repair_specs
         - **Silt Accumulation (> 30% reduction in drain dep

## Part 3 — Test the LangGraph Agentic RAG Workflow

Run the full stateful graph on real municipal repair scenarios:
1. **Scenario A**: High-Pressure Water Pipe Rupture (Critical)
2. **Scenario B**: Deep Road Pothole Cluster on Highway (Poor/Urgent)

In [4]:
# Initialize the compiled LangGraph workflow
graph = create_cost_estimator_graph()

# Scenario A: Water Pipe Rupture
scenario_a = {
    "messages": [],
    "hazard_type": "Pipe Burst",
    "severity": "Critical",
    "asset_name": "Main St Water Pipe (AST-001)",
    "asset_type": "Water",
    "damage_description": "Major 110mm uPVC pipe burst near junction causing severe flooding and pavement washaway.",
    "location": "Downtown Colombo",
    "search_query": "",
    "retrieved_docs": "",
    "retries": 0,
    "intent": "estimate",
    "is_relevant": False,
    "estimate": None,
    "final_response": "",
}

print("Running LangGraph Agent on Scenario A...")
config = {"configurable": {"thread_id": "test-scenario-a"}}
result_a = graph.invoke(scenario_a, config=config)

print("\n" + "#" * 60)
print("RESULT: SCENARIO A (WATER PIPE BURST)")
print("#" * 60)
print(result_a.get("final_response"))

est_a = result_a.get("estimate")
if est_a:
    print("\n--- Itemized Materials Breakdown (LKR) ---")
    for m in est_a.get("materials", []):
        print(f"  * {m['item_name']:<45} : {m['quantity']} {m['unit']} @ LKR {m['unit_cost_lkr']:,.2f} = LKR {m['total_cost_lkr']:,.2f}")
    print("\n--- Plant & Labor Breakdown (LKR) ---")
    for l in est_a.get("labor_and_equipment", []):
        print(f"  * {l['role_or_machine']:<45} : {l['days']} days @ LKR {l['daily_rate_lkr']:,.2f} = LKR {l['total_cost_lkr']:,.2f}")
    print(f"\nSafety & Preliminaries : LKR {est_a.get('safety_and_preliminaries_lkr', 0):,.2f}")
    print(f"Contingency ({est_a.get('contingency_percentage')}%)     : LKR {est_a.get('contingency_cost_lkr', 0):,.2f}")
    print(f"GRAND TOTAL BUDGET      : LKR {est_a.get('total_estimated_cost_lkr', 0):,.2f}")

Running LangGraph Agent on Scenario A...


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



############################################################
RESULT: SCENARIO A (WATER PIPE BURST)
############################################################
### Cost & Material Estimate: Main St Water Pipe (AST-001)

**Summary**: Critical repair of a major 110mm uPVC pipe burst on Main St, Downtown Colombo, causing significant flooding and pavement damage. The scope includes excavation, replacement of the damaged pipe section, jointing, pressure testing, backfilling, and temporary reinstatement of the affected pavement area. Permanent road reinstatement will follow.

**Category**: Water | **Severity**: Critical | **Duration**: 4 days

**Total Budget**: **LKR 253,320.00**

**Recommended Contractor**: Water & Plumbing, Civil

**Technical Notes**: Excavation must be carefully executed to prevent damage to adjacent utilities. Pipe laying and jointing shall strictly adhere to NWSDB standards for uPVC pipes, including proper bedding and haunching. A mandatory pressure test of the new pip

In [5]:
# Scenario B: Road Pothole Cluster
scenario_b = {
    "messages": [],
    "hazard_type": "Pothole Cluster",
    "severity": "Poor",
    "asset_name": "Galle Rd Bridge Approach (AST-004)",
    "asset_type": "Roads & Bridges",
    "damage_description": "Deep pothole cluster (approx 15 m2 area, 60mm depth) with asphalt crumbling near bridge expansion joint.",
    "location": "Colombo 03",
    "search_query": "",
    "retrieved_docs": "",
    "retries": 0,
    "intent": "estimate",
    "is_relevant": False,
    "estimate": None,
    "final_response": "",
}

print("Running LangGraph Agent on Scenario B...")
config = {"configurable": {"thread_id": "test-scenario-b"}}
result_b = graph.invoke(scenario_b, config=config)

print("\n" + "#" * 60)
print("RESULT: SCENARIO B (ROAD POTHOLES)")
print("#" * 60)
print(result_b.get("final_response"))

est_b = result_b.get("estimate")
if est_b:
    print(f"\nGrand Total Budget : LKR {est_b.get('total_estimated_cost_lkr', 0):,.2f}")
    print(f"Turnaround Time    : {est_b.get('estimated_duration_days')} days")
    print(f"Contractor Trade   : {est_b.get('recommended_contractor_specialization')}")

Running LangGraph Agent on Scenario B...

############################################################
RESULT: SCENARIO B (ROAD POTHOLES)
############################################################
### Cost & Material Estimate: Galle Rd Bridge Approach (AST-004)

**Summary**: Executive summary: Repair of a 15 m² deep pothole cluster on Galle Rd Bridge Approach (AST-004) in Colombo 03. The intervention involves thorough surface preparation, application of bitumen emulsion tack coat, and placement of a 60mm compacted hot-mix asphalt binder course to restore the road surface integrity. The estimated cost is LKR 278,025 with a 2-day turnaround.

**Category**: Roads & Bridges | **Severity**: Poor | **Duration**: 2 days

**Total Budget**: **LKR 278,025.00**

**Recommended Contractor**: Roads & Bridges

**Technical Notes**: Prior to asphalt placement, the pothole area must be thoroughly swept and cleaned to remove all dust, debris, and standing water. A cationic slow-setting bitumen emulsion

## Part 4 — Test FastAPI HTTP Service Directly

Test the REST endpoints that your React frontend (`CiviLanka.Web`) will connect to:
- `GET  /health`
- `GET  /api/agent/search?query=...`
- `POST /api/agent/estimate`

In [6]:
API_URL = "http://127.0.0.1:8001"

try:
    # 1. Test Health
    res = requests.get(f"{API_URL}/health", timeout=3)
    print("Health Endpoint Response:")
    print(json.dumps(res.json(), indent=2))
    print("\n[SUCCESS] FastAPI server is reachable on port 8001!")
except Exception as ex:
    print(f"[Notice] FastAPI server is not running on {API_URL}.")
    print("To start it in a terminal, run:\n  cd CiviLanka.Agent\n  uvicorn main:app --reload --port 8001")

[Notice] FastAPI server is not running on http://127.0.0.1:8001.
To start it in a terminal, run:
  cd CiviLanka.Agent
  uvicorn main:app --reload --port 8001


In [7]:
# 2. Test POST /api/agent/estimate via HTTP
payload = {
    "hazard_type": "Pipe Burst",
    "severity": "Critical",
    "asset_name": "Main St Water Pipe",
    "asset_type": "Water",
    "damage_description": "High-pressure 110mm water pipe ruptured under main street pavement.",
    "location": "Downtown Colombo"
}

try:
    res = requests.post(f"{API_URL}/api/agent/estimate", json=payload, timeout=10)
    data = res.json()
    print("API Estimate Response Status:", res.status_code)
    print("Asset Name                 :", data.get("asset_name"))
    print("Severity                   :", data.get("severity"))
    print("Total Cost (LKR)           : LKR", f"{data['estimate']['total_estimated_cost_lkr']:,.2f}")
    print("Duration                   :", f"{data['estimate']['estimated_duration_days']} days")
    print("Contractor                 :", data['estimate']['recommended_contractor_specialization'])
    print("\n[SUCCESS] Full API flow verified and ready for frontend integration!")
except Exception as ex:
    print("HTTP Call skipped or server offline:", ex)

HTTP Call skipped or server offline: HTTPConnectionPool(host='127.0.0.1', port=8001): Max retries exceeded with url: /api/agent/estimate (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8001): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))
